# FantasyVoice — 한국어 샘플 전사와 청취 검수
Voice 원본은 변경하지 않습니다. 이 노트북은 모델 학습을 실행하지 않습니다.
GPU 런타임을 선택하세요. dist/fantasyvoice-pilot.zip을 MyDrive/FantasyVoice/에 올린 후 실행합니다.
Google Drive 연결은 본인이 승인해야 합니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import sys, json, subprocess, zipfile, platform
VOICE_ROOT = Path('/content/drive/MyDrive/Voice')
BUNDLE = Path('/content/drive/MyDrive/FantasyVoice/fantasyvoice-pilot.zip')
PROJECT = Path('/content/fantasyvoice-pilot')
OUTPUT = Path('/content/drive/MyDrive/FantasyVoice/pilot-v1')
MODEL_CACHE = Path('/content/whisper-models')
INVENTORY_MODE = 'reference'  # Drive 전체 검사가 필요할 때만 'scan'
PILOT_LIMIT = 12  # 0이면 전체 파일럿 목록. 첫 청취 검증용 수량이며 학습 기준이 아님.
if not VOICE_ROOT.is_dir():
    raise FileNotFoundError(VOICE_ROOT)
if not BUNDLE.is_file():
    raise FileNotFoundError(f'먼저 코드 ZIP을 Drive에 올려주세요: {BUNDLE}')
with zipfile.ZipFile(BUNDLE) as archive:
    for item in archive.infolist():
        destination = (PROJECT / item.filename).resolve()
        if not destination.is_relative_to(PROJECT.resolve()):
            raise ValueError('Unsafe archive path')
    archive.extractall(PROJECT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e',
                str(PROJECT) + '[asr]', 'ipywidgets'], check=True)
# Editable package 경로를 이미 시작한 kernel에서도 사용할 수 있도록 설정
sys.path.insert(0, str(PROJECT / 'src'))
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Colab 런타임을 GPU로 바꾼 후 다시 실행하세요.')
from fantasyvoice.dataset.storage import check_output, write_json, write_jsonl, read_jsonl
check_output(VOICE_ROOT, OUTPUT)
OUTPUT.mkdir(parents=True, exist_ok=True)
runtime = {'python': platform.python_version(), 'torch': str(torch.__version__),
           'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0),
           'vram_gb': round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2)}
print(runtime)
from datetime import datetime, timezone
write_json(OUTPUT / ('runtime-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S') + '.json'), runtime)


## 원본 목록과 파일럿 선택
기본 reference 모드는 ZIP에 포함한 로컬 목록을 즉시 불러옵니다. Drive 전체 파일을 읽지 않습니다.
표시하는 총 개수는 로컬 목록 기준이며 Drive 업로드 완료를 보증하지 않습니다.
선택된 음성은 전사 직전에 Drive에서 SHA256을 확인합니다. 누락/변경 파일은 오류로 기록합니다.
전체 Drive 내용을 새로 조사하려면 INVENTORY_MODE를 scan으로 바꾸세요. 많은 시간이 걸릴 수 있습니다.
캐릭터별 길이가 다른 표본과 공격 계열 표본을 선택하며, 학습 합격/불합격 판정은 하지 않습니다.

0.5초 이하 음성은 후보에서 제외합니다. 0.5초를 넘어도 비언어 발성일 수 있어 검수는 필요합니다.


In [ ]:
from fantasyvoice.dataset.inventory import scan, select_pilot, summarize, load_reference
if INVENTORY_MODE == 'reference':
    rows = load_reference(PROJECT / 'reference/inventory.jsonl')
    print('로컬 기준 목록 사용: Drive 전체 스캔 생략. 선택 파일은 전사 전 해시 검증.')
elif INVENTORY_MODE == 'scan':
    rows = scan(VOICE_ROOT, lambda n, total: print(f'목록 {n}/{total}', flush=True))
else:
    raise ValueError('INVENTORY_MODE must be reference or scan')
write_json(OUTPUT / 'inventory-origin.json', {
    'mode': INVENTORY_MODE,
    'remote_full_inventory_verified': INVENTORY_MODE == 'scan',
    'selected_audio_hash_check': 'performed by run_pilot before transcription'
})
write_jsonl(OUTPUT / 'inventory.jsonl', rows)
summary = summarize(rows)
summary['inventory_origin'] = INVENTORY_MODE
if INVENTORY_MODE == 'reference':
    summary['validation'] = 'local_reference_only_not_remote_full_verification'
write_json(OUTPUT / 'summary.json', summary)
duration_excluded = [r for r in rows if r['status'] == 'ok' and r['duration_seconds'] <= 0.5]
write_jsonl(OUTPUT / 'excluded-duration.jsonl', duration_excluded)
print('0.5초 이하 제외:', len(duration_excluded), '개 (원본 보존)')
candidates = select_pilot(rows, per_character=3)
# 작은 파일럿에서도 한 캐릭터에 몰리지 않도록 순환 선택
from collections import defaultdict
groups = defaultdict(list)
for row in candidates:
    groups[row['character_id']].append(row)
ordered = []
while any(groups.values()):
    for name in sorted(groups):
        if groups[name]:
            ordered.append(groups[name].pop(0))
if not isinstance(PILOT_LIMIT, int) or PILOT_LIMIT < 0:
    raise ValueError('PILOT_LIMIT는 0 이상의 정수여야 합니다.')
selection = ordered[:PILOT_LIMIT] if PILOT_LIMIT else ordered
if not selection:
    raise ValueError('읽을 수 있는 샘플이 없습니다.')
write_jsonl(OUTPUT / 'pilot.jsonl', selection)
print('목록 기준 총 파일:', len(rows), '헤더 오류:', sum(r['status'] != 'ok' for r in rows))
for row in selection:
    print(row['audio_path'], round(row['duration_seconds'], 3), '초', row['subtype'])
print('위 목록을 확인한 뒤 다음 전사 셀을 실행하세요.')


## Whisper large-v3 전사
최초 실행 시 모델을 다운로드합니다. 다른 모델로 자동 전환하지 않습니다.
동일 설정으로 재실행하면 완료 파일은 재사용하고 실패 파일은 재시도합니다.
출력은 자동 전사 후보이며 모두 청취 검수를 기다립니다.


In [ ]:
from fantasyvoice.dataset.transcribe import WhisperBackend, run_pilot
options = json.loads((PROJECT / 'config/pilot.json').read_text(encoding='utf-8'))['whisper_options']
print('전사 옵션:', options)
backend = WhisperBackend(options, MODEL_CACHE)
results = run_pilot(VOICE_ROOT, selection, OUTPUT, backend.config, backend,
                    lambda n, total, state: print(f'전사 {n}/{total}: {state}', flush=True))
latest = {r['key']: r for r in results}
print('성공:', sum(r['status'] == 'success' for r in latest.values()))
print('오류:', sum(r['status'] == 'error' for r in latest.values()))
for row in latest.values():
    if row['status'] == 'error':
        print(row['audio_path'], row['error'])
del backend
torch.cuda.empty_cache()


## 청취 후 전사 수정·판정
재생 버튼으로 음성을 듣고 수정문과 판정을 저장하세요.
재생은 원래 음량을 유지합니다. PCM 범위를 넘는 신호만 재생용으로 줄이며 원본은 변경하지 않습니다. 무음/빈 파일은 재생 대신 상태를 표시합니다.
accepted는 **대사 확인**을 뜻하며 최종 학습 데이터 합격 판정은 아닙니다.
실패/빈 전사, 기합, 웃음은 실제 음성을 듣고 판단하세요.


In [ ]:
import ipywidgets as widgets
import soundfile as sf
from IPython.display import display, Audio, clear_output
from fantasyvoice.dataset.review import save_review, playback_audio
from fantasyvoice.dataset.storage import resolve_audio, sha256
latest = {r['key']: r for r in read_jsonl(OUTPUT / 'transcripts.jsonl')}
selected_paths = {r['audio_path'] for r in read_jsonl(OUTPUT / 'pilot.jsonl')}
successful = {key: row for key, row in latest.items()
              if row['status'] == 'success' and row['audio_path'] in selected_paths}
if not successful:
    raise RuntimeError('검수할 성공 전사가 없습니다. 위 오류를 먼저 확인하세요.')
picker = widgets.Dropdown(options=[(r['audio_path'], key) for key, r in successful.items()],
                          description='파일', layout=widgets.Layout(width='95%'))
text = widgets.Textarea(description='수정 대사', layout=widgets.Layout(width='95%', height='100px'))
decision = widgets.Dropdown(options=[('보류: 판단 미완료', 'pending'), ('대사 확인: 수정문과 음성이 일치', 'accepted'), ('부적합: 학습용으로 사용 안 함', 'rejected')])
note = widgets.Text(description='메모', layout=widgets.Layout(width='95%'))
save = widgets.Button(description='검수 저장', button_style='primary')
player, feedback = widgets.Output(), widgets.Output()
def show(change=None):
    row = successful[picker.value]
    # Update editable fields first so a playback error cannot leave another file's text.
    history = {r['key']: r for r in read_jsonl(OUTPUT / 'reviews.jsonl')}
    previous = history.get(picker.value)
    text.value = previous['corrected_text'] if previous else row['asr']['text']
    decision.value = previous['decision'] if previous else 'pending'
    note.value = previous['note'] if previous else ''
    save.disabled = True
    with feedback:
        clear_output()
    with player:
        clear_output(wait=True)
        print('자동 전사:', row['asr']['text'])
        print('검수 사유:', row['review_reasons'])
        try:
            path = resolve_audio(VOICE_ROOT, row['audio_path'])
            if sha256(path) != row['sha256']:
                raise ValueError('원본이 바뀌었습니다. 목록/전사를 갱신해주세요.')
            save.disabled = False
            preview = playback_audio(path)
            print(f"길이: {preview['duration_seconds']:.3f}초")
            if preview['status'] == 'silent':
                print('완전한 무음 파일입니다. 학습용 대사가 없으므로 부적합으로 검수하세요.')
            elif preview['status'] == 'empty':
                print('샘플이 없는 빈 파일입니다. 부적합으로 검수하세요.')
            else:
                display(Audio(data=preview['wav_bytes']))
                if preview['playback_gain'] < 1:
                    print('PCM 범위를 넘는 신호여서 재생용 음량만 줄였습니다.')
        except Exception as exc:
            print('재생 준비 실패:', type(exc).__name__, str(exc))
            print('다른 파일을 선택할 수 있습니다. 불명확하면 보류로 기록하세요.')
def persist(button):
    with feedback:
        clear_output()
        try:
            row = successful[picker.value]
            if sha256(resolve_audio(VOICE_ROOT, row['audio_path'])) != row['sha256']:
                raise ValueError('원본이 바뀌었습니다. 다시 전사하세요.')
            save_review(OUTPUT, picker.value, decision.value, text.value, note.value)
            print('저장됨:', row['audio_path'], decision.value)
        except Exception as exc:
            print('저장 실패:', exc)
picker.observe(show, names='value')
save.on_click(persist)
display(picker, player, text, decision, note, save, feedback)
show()


## 결과 확인
결과는 설정 셀의 OUTPUT에 있습니다. 같은 결과 디렉터리를 여러 런타임에서 동시에 사용하지 마세요.
다음 단계에서는 청취 결과를 보고 전사 검수 기준, 스타일 추출 설정과 학습 데이터 분할을 정합니다.


In [ ]:
reviews = read_jsonl(OUTPUT / 'reviews.jsonl')
current = {r['key']: r for r in reviews}
from collections import Counter
print('검수 현황:', dict(Counter(r['decision'] for r in current.values())))
print('결과 폴더:', OUTPUT)
print('자동 전사:', OUTPUT / 'transcripts.jsonl')
print('검수 이력:', OUTPUT / 'reviews.jsonl')
